In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
## general libraries
import pathlib
from rich.pretty import install, pprint
from loguru import logger

## data handling libraries
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.dask import TqdmCallback as ProgressBarDask

# plotting libraries
import matplotlib.pyplot as plt
import hvplot.xarray
import panel

## machine learning libraries
import gpytorch 
import torch
import ngboost
from sklearn import metrics
from sklearn import ensemble as trees

## PAMIR libraries
import mlflow
import hydra
import ngboost
import pamir_mlpermafrost as pamir
from cryogrid_pytools import xr_raster_vector

ngb = pamir.models.ngb

install(overflow=True)

In [3]:
cfg = pamir.utils.load_hydra_config(
    config_dir='../../src/pamir_mlpermafrost/conf', 
    config_name='ngboost-multi')

2025-09-24 12:35:38 | WARNING  - Will pop `run_dir` since detected in Notebook and cant be resolved


In [4]:
data = ngb.main.load_training_data(cfg)
train = data.train
valid = data.valid
test = data.test

n_targets = data.train.y.shape[1]
f"{n_targets=}"

'n_targets=12'

In [5]:
model = ngboost.NGBRegressor(
    Dist=ngboost.distns.MultivariateNormal(n_targets),
    Base=trees.GradientBoostingRegressor(
        max_depth=12, 
        min_samples_leaf=12),
    learning_rate=0.01,
    verbose_eval=1,
    n_estimators=2000,
)

In [6]:
model.fit(train.x, train.y, valid.x, valid.y, early_stopping_rounds=20)

[iter 0] loss=-3.3319 val_loss=-2.9428 scale=0.0156 norm=4.0120
[iter 1] loss=-3.3425 val_loss=-2.9435 scale=0.0156 norm=4.0029
[iter 2] loss=-3.3530 val_loss=-2.9445 scale=0.0156 norm=3.9936
[iter 3] loss=-3.3634 val_loss=-2.9453 scale=0.0156 norm=3.9849


KeyboardInterrupt: 

In [24]:
yhat_dist = model.pred_dist(train.x)

In [52]:
yhat_dist.loc


array([[ 7.38174449,  6.57311114],
       [ 5.36892505,  3.46476806],
       [ 6.13024575,  4.67104581],
       ...,
       [10.0777165 ,  9.31477598],
       [10.02988646,  9.12972487],
       [ 7.56956329,  7.12627411]], shape=(5311, 2))

In [103]:
dists = yhat_dist.scipy_distribution()